# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [2]:
import os
import sys

# so that mllm_shap can be imported without installing the package
sys.path.insert(0, os.path.abspath("../mllm_shap/src"))

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["LOG_LEVEL"] = "INFO"

In [3]:
import numpy as np
import pandas as pd
import torch

np.random.seed(42)

device = (
    torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
print(f"Using device: {device}")

/Users/pawel.pozorski/Desktop/MLLM-Shap/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using device: mps


In [4]:
from mllm_shap.connectors import LiquidAudio, ModelConfig
from mllm_shap.connectors.enums import ModelHistoryTrackingMode, Role, SystemRolesSetup
from mllm_shap.connectors.filters import KeepAllTokens
from mllm_shap.utils.jupyter import display_shap_colors_df
from mllm_shap.shap import Explainer, ComplementaryShapExplainer

Define LiquidAudio model (this call loads it up to the memory!).

In [5]:
model = LiquidAudio(
    device=device, history_tracking_mode=ModelHistoryTrackingMode.TEXT
)  # track and generate only text history

W0505 12:59:32.834000 75658 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Create compact explainer that will make initial call and then explain it using shapley values using Complementary Formula.

In [6]:
explainer = Explainer(
    model=model, shap_explainer=ComplementaryShapExplainer(fraction=0.015)
)

Create new chat instance and assign it messages.

In [7]:
chat = model.get_new_chat(
    system_roles_setup=SystemRolesSetup.NONE,  # calculate shapley values for all roles
    token_filter=KeepAllTokens(),  # keep all tokens for shapley values calculation
)

chat.new_turn(Role.USER)
chat.add_text("Who are you?")
chat.end_turn()

# Usage

Let's calculate shapley values for current conversation.

Generation kwargs allows to customize model interference - here we limit it to 4 tokens and change text_temperature from default 0.0 to 0.2, text_top_k from default 1 to 3. 

In [8]:
generation_kwargs = {
    "max_new_tokens": 4,
    "model_config": ModelConfig(text_temperature=0.2, text_top_k=3),
}

result = explainer(
    chat=chat,
    generation_kwargs=generation_kwargs,
    progress_bar=True,  # show progress bar during generation, default
)

2026-05-05 12:59:37,580 - mllm_shap.shap.compact - INFO - Generating full response from the model...
2026-05-05 12:59:39,797 - mllm_shap.shap.base._masks_manager - INFO - Number of tokens for explainability: 10 (up to 1023 additional calls)
2026-05-05 12:59:39,825 - mllm_shap.shap.complementary._approximation - WARNING - Calculated number of samples (20) is less than minimal required (20). Using minimal number of samples.


Complementary SHAP:   0%|          | 0/20 [00:00<?, ?it/s]

2026-05-05 12:59:42,041 - mllm_shap.shap.complementary._approximation - WARNING - Calculated number of samples (20) is less than minimal required (20). Using minimal number of samples.
2026-05-05 12:59:44,685 - mllm_shap.shap.base._generate_responses - INFO - Generation stats: processed=20 cache_hits=0 cache_misses=20 skipped_filtered=0 model_elapsed_ms=4543.70
2026-05-05 12:59:44,685 - mllm_shap.shap.base.shap_explainer - INFO - Sampling stats: candidates=10 yielded=20 skipped(full_or_empty)=0 skipped(invalid)=0 skipped(duplicates)=0 elapsed_ms=4855.98


Let's see final Shap values.

In [9]:
display_shap_colors_df(
    pd.DataFrame(
        list(
            zip(
                [chat.decode_text(token) for token in result.full_chat.input_tokens],
                result.full_chat.cache.normalized_values.tolist(),
            )
        ),
        columns=["Text", "Shapley Value"],
    )
)

,Text,Shapley Value
0,<|startoftext|>,0.153320
1,<|im_start|>,0.052979
2,user,0.092773
3,,0.143555
4,Who,0.163086
5,are,0.000000
6,you,0.202148
7,?,0.064941
8,<|im_end|>,0.094727
9,,0.033203


# Tests

In [10]:
explainer.shap_explainer._C

tensor([[ 0.0000,  0.0000,  0.0000, -0.1719, -0.1602,  0.2266,  0.3945,  0.2656,
          0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000, -0.4492, -0.2266,  0.1055,  0.4375,
          0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000, -0.0078, -0.2266,  0.5469,  0.4375,
          0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000, -0.6016,  0.2266, -0.0469,  0.4375,
          0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000, -0.3867,  0.2266,  0.1680,  0.4375,
          0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000, -0.4375,  0.0547, -0.2266,  0.6094,  0.0000,
          0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000, -0.1719, -0.1172,  0.2266,  0.4375,  0.2656,
          0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000, -0.1992, -0.2266,  0.3555,  0.4375,
          0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000, -0.2656, -0.4062,  0

In [11]:
explainer.shap_explainer._M

tensor([[0, 0, 0, 1, 2, 1, 5, 1, 0, 0, 0],
        [0, 0, 0, 0, 4, 1, 3, 2, 0, 0, 0],
        [0, 0, 0, 0, 1, 1, 6, 2, 0, 0, 0],
        [0, 0, 0, 0, 4, 1, 3, 2, 0, 0, 0],
        [0, 0, 0, 0, 3, 1, 4, 2, 0, 0, 0],
        [0, 0, 0, 2, 2, 1, 5, 0, 0, 0, 0],
        [0, 0, 0, 1, 5, 1, 2, 1, 0, 0, 0],
        [0, 0, 0, 0, 2, 1, 5, 2, 0, 0, 0],
        [0, 0, 0, 1, 4, 1, 3, 1, 0, 0, 0],
        [0, 0, 0, 1, 1, 1, 6, 1, 0, 0, 0]], device='mps:0', dtype=torch.int16)